# 02 — Train encoders on GPU (transductive, inductive, learning curve)

Each run is resumable (epoch checkpoints on Drive) and never retrained once finished. Families are trained one by one; ablations are opt-in. TRANSDUCTIVE encoders serve label-free diagnostics only; INDUCTIVE encoders (one per outer fold, held-out athletes removed from both domains) are the only ones used for NFL prediction.

In [ ]:
%pip install -q "git+https://github.com/YOUR_USERNAME/YOUR_REPO.git"
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from nflbowl import Experiment

exp = Experiment(data_dir="/content/drive/MyDrive/NFLBowl/bundles/m12f_v1", output_dir="/content/drive/MyDrive/NFLBowl",
                 config="colab_gpu", name="f4_main")
exp.prepare()
exp.list_models()

In [ ]:
# primary encoder, then the other families (ablations=False by default)
exp.train("F4", resume=True)
exp.train_all(families=True, ablations=False)

In [ ]:
# ablations + negative controls (F4_nophys, F4_noalign, F4_adv, F4_mmd, F4_category, F4_shuffled_phys) — optional
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    exp.train_all(families=False, ablations=True)

In [ ]:
# inductive encoders: F2 and F4 × outer folds (science.inductive_configs)
exp.train_all(regime="inductive")

In [ ]:
# learning curve against unlabeled NFL data (transductive F4 with 10 % and 30 % of the NFL windows)
for frac in exp.cfg.science.learning_curve_fractions:
    exp.train("F4", nfl_fraction=frac)

In [ ]:
# after a disconnect: continue every unfinished job
exp.resume()
exp.list_artifacts("training")